# Class 1 - What Makes an Agent an Agent?

**Week 5: Introduction to AI Agents**

### Learning objectives
By the end of this notebook you will be able to:
- Explain how an agent differs from a single prompt/response.
- Describe autonomy as a spectrum (low / medium / high).
- Walk a manual Plan → Act → Observe chain in plain Python + Groq.
- Spot agent-like behavior in short scenarios.

Run cells in order with **Shift+Enter**. You need a `GROQ_API_KEY` for the live demo cells (Colab secret or env var). Classification exercises run without a key.

## Setup

Install the Groq SDK, then resolve your API key. Never hardcode a key in a notebook.

```bash
export GROQ_API_KEY="gsk-..."
```

In Colab: add a secret named `GROQ_API_KEY` and enable notebook access.

In [1]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:00


In [2]:
import os

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
except Exception:
    pass

if not GROQ_API_KEY:
    print(
        "No GROQ_API_KEY found.\n"
        "Set it in your environment or add a Colab secret named GROQ_API_KEY.\n"
        "Live demo cells below will skip until a key is available."
    )
else:
    print("Found GROQ_API_KEY. You're ready to run the demo cells.")


def ask(system_prompt, user_prompt, model="llama-3.3-70b-versatile", temperature=0.2, max_tokens=400):
    """Send a system + user message to Groq. Returns assistant text, or None if no key."""
    if not GROQ_API_KEY:
        print("Skipping live call — no GROQ_API_KEY set.")
        return None
    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY)
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return resp.choices[0].message.content

Found GROQ_API_KEY. You're ready to run the demo cells.


## 1. A single prompt vs an agent-shaped task

A single completion is one shot: question in, answer out. Agents keep going — they can decide on a next action, use a tool or intermediate step, observe the result, and continue until the goal is met.

Below we ask a multi-step question in one shot. Notice what the model invents vs what it would need to look up.

In [4]:
single_shot_question = (
    "I have a flight landing in Kathmandu at 18:00 local time. "
    "Should I pack a jacket tonight, and what is 14C in Fahrenheit?"
)

reply = ask(
    system_prompt="You are a helpful assistant. Answer briefly.",
    user_prompt=single_shot_question,
)
print(reply or "(no reply — set GROQ_API_KEY to run this cell)")

Kathmandu can get cool in the evenings. It's a good idea to pack a jacket. 
14C is approximately 57F.


## 2. A manual Plan → Act → Observe chain

We simulate an agent without a framework. Each step is a separate Groq call you control:

1. **Plan** — decide what information we need.
2. **Act** — "look up" weather with a fake tool (a Python dict).
3. **Observe** — feed the tool result back and ask for a final answer.

This is the same loop Class 2 will automate with built-in tools.

In [5]:
MOCK_WEATHER = {
    "Kathmandu": {"temp_c": 14, "condition": "rain"},
    "Pokhara": {"temp_c": 18, "condition": "clear"},
}


def fake_get_weather(city: str) -> dict:
    """Stand-in for a real weather API — the Act step."""
    return MOCK_WEATHER.get(city, {"temp_c": None, "condition": "unknown"})


def manual_chain(city: str):
    # Plan
    plan = ask(
        system_prompt="You plan tool use. Reply with one city name to look up and nothing else.",
        user_prompt=f"User wants jacket advice for tonight near {city}. Which city should we query?",
    )
    print("PLAN:", plan)

    # Act (our code runs the tool — not the model)
    lookup_city = city  # keep deterministic for the demo
    observation = fake_get_weather(lookup_city)
    print("ACT -> fake_get_weather:", observation)

    # Observe
    final = ask(
        system_prompt="You give packing advice. Use only the weather observation provided.",
        user_prompt=(
            f"City: {lookup_city}. Observation: {observation}. "
            "Should the traveler bring a jacket? One short paragraph."
        ),
    )
    print("OBSERVE / FINAL:", final)
    return final


manual_chain("Kathmandu")

PLAN: Kathmandu
ACT -> fake_get_weather: {'temp_c': 14, 'condition': 'rain'}
OBSERVE / FINAL: Given the current weather observation in Kathmandu, with a temperature of 14 degrees Celsius and rain, it's likely to be quite chilly and wet. In this case, yes, the traveler should consider bringing a jacket to stay warm and dry. A waterproof or water-resistant jacket would be a good choice to protect against the rain, and its insulating properties will help to keep the traveler warm in the relatively cool temperature.


"Given the current weather observation in Kathmandu, with a temperature of 14 degrees Celsius and rain, it's likely to be quite chilly and wet. In this case, yes, the traveler should consider bringing a jacket to stay warm and dry. A waterproof or water-resistant jacket would be a good choice to protect against the rain, and its insulating properties will help to keep the traveler warm in the relatively cool temperature."

## 3. Autonomy is a spectrum

Not every LLM app is an agent. Use these labels:

- **Low** — model replies once; human does all follow-up.
- **Medium** — model may call tools / take a few steps inside a fixed loop.
- **High** — model pursues a goal over many steps with little supervision.

Classify each scenario by writing `low`, `medium`, or `high` next to it.

In [7]:
SCENARIOS = [
    "A FAQ bot answers one customer question from a system prompt, then stops.",
    "A coding assistant edits files, runs tests, and retries until tests pass (with a human approving merges).",
    "A travel helper calls get_weather then answers whether to bring a jacket.",
    "An overnight research agent browses the web for hours and emails a report with no check-ins.",
    "Autocomplete suggests the next few tokens in an IDE as you type.",
]

# Fill in your labels: "low" | "medium" | "high"
your_labels = {
    SCENARIOS[0]: "low",  # TODO
    SCENARIOS[1]: "high",
    SCENARIOS[2]: "medium",
    SCENARIOS[3]: "high",
    SCENARIOS[4]: "low",
}

for s, label in your_labels.items():
    print(f"[{label or '?'}] {s}")

[?] A FAQ bot answers one customer question from a system prompt, then stops.
[?] A coding assistant edits files, runs tests, and retries until tests pass (with a human approving merges).
[?] A travel helper calls get_weather then answers whether to bring a jacket.
[?] An overnight research agent browses the web for hours and emails a report with no check-ins.
[?] Autocomplete suggests the next few tokens in an IDE as you type.


## Closing

You now have a working definition: agents pursue a goal through a loop of decisions and actions. Next class, Groq plus an agent framework will own the Plan → Act → Observe wiring so you are not hand-rolling every step.

**Next:** Class 2 — The Agent Loop (tool wiring).

## Challenges

Complete each challenge in the TODO cells. Do not peek at Class 2 yet — stay on plain Groq / Python.

### Challenge 01 — Change the one-shot question
Ask a different one-sentence multi-step question with `ask(...)` and print the reply.

In [ ]:
# TODO: call ask(...) with your own one-sentence question
pass

In [9]:
reply = ask(system_prompt="You are a helpful assistant.", user_prompt="What is your favorite food, why do you like it, and where would you eat it most?")
print(reply)

As a digital assistant, I don't have personal preferences, taste buds, or a physical presence, so I don't have a favorite food. I exist solely to provide information and assist with tasks, but I don't have personal experiences or emotions.

However, I can provide information about different types of cuisine, recipes, and restaurants if that's helpful! If you have a specific type of food in mind or a particular cuisine you're interested in, I'd be happy to help you explore it. Just let me know how I can assist you.


### Challenge 02 — Write a 3-step manual chain from scratch
Pick a new fake tool (e.g. `fake_fx_rate(from_currency, to_currency)`) and run Plan → Act → Observe yourself.

In [1]:
# TODO: define a fake tool dict + manual_chain-style function, then run it
pass

In [2]:
# TODO: define a fake tool dict + manual_chain-style function, then run it

tools = {
    "add": lambda a, b: a + b,
    "multiply": lambda a, b: a * b
}

def manual_chain():
    result1 = tools["add"](5, 3)
    result2 = tools["multiply"](result1, 2)
    return result2

print(manual_chain())

16


### Challenge 03 — Classify five scenarios
Finish the `your_labels` dict in Section 3 (or copy it here) so every scenario has `low`, `medium`, or `high`.

In [ ]:
# TODO: paste/complete your_labels and print them
pass

In [3]:
your_labels = ["positive", "negative", "neutral"]

print(your_labels)

['positive', 'negative', 'neutral']


### Challenge 04 (stretch) — Spot the agent
Write two short transcripts (3–5 lines each). Mark which one is more agent-like and why (2–3 sentences).

In [ ]:
# TODO: write transcript_a, transcript_b, and a short justification string
pass

In [4]:
transcript_a = "The customer was happy with the service."
transcript_b = "The customer was satisfied with the service."

justification = "Both transcripts have the same meaning, although the wording is slightly different."

print(transcript_a)
print(transcript_b)
print(justification)

The customer was happy with the service.
The customer was satisfied with the service.
Both transcripts have the same meaning, although the wording is slightly different.
